# MCP (Model Context Protocol) Server Implementation

In [1]:
!python -m pip install fastmcp langchain_mcp_adapters --quiet

In [2]:
import os
os.makedirs("mcpserver",exist_ok=True)

In [3]:
%%writefile mcpserver/mcpserver1.py

from mcp.server.fastmcp import FastMCP
import requests, json
import wikipedia

mcp = FastMCP("TredenceMCP")

@mcp.tool()
async def get_current_weather(city:str)->dict:
    """ this funciton can be used to get current weather information"""
    api_key="6a8b0ac166a37e2b7a38e64416b3c3fe"
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}"
    response = requests.get(url)
    response = json.loads(response.content.decode())
    output = {"city":city,"weather":response['weather'][0]['description'],
              "temperature":response['main']['temp'], "unit":"kelvin"
              }
    return output

@mcp.tool()
async def get_wikipedia_summary(query:str)->str:
    response = wikipedia.summary(query)
    return response


if __name__=="__main__":
    mcp.run(transport='streamable-http')


Writing mcpserver/mcpserver1.py


In [9]:
# run mCP server: python mcpserver/mcpserver1.py

## Implement an Agent which connects to the MCP server and fetches the tools

In [4]:
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

In [6]:
client = MultiServerMCPClient({"TredenceMCP":{"url":"http://127.0.0.1:8000/mcp",
                                              "transport":"streamable_http"}})


tools = await client.get_tools()
tools

[StructuredTool(name='get_current_weather', description=' this funciton can be used to get current weather information', args_schema={'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_current_weatherArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7c4a22eeb740>),
 StructuredTool(name='get_wikipedia_summary', args_schema={'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'get_wikipedia_summaryArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7c4a22eebec0>)]

In [7]:
agent = create_agent("google_genai:gemini-2.0-flash",tools)
await agent.ainvoke({"messages":[{"role":'user',"content":"what is the weather in Delhi?"}]})

{'messages': [HumanMessage(content='what is the weather in Delhi?', additional_kwargs={}, response_metadata={}, id='70dd32ff-c24f-4025-8dd9-7861093b90d2'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_weather', 'arguments': '{"city": "Delhi"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--48072a53-20aa-4f79-bbf0-53806eb959b4-0', tool_calls=[{'name': 'get_current_weather', 'args': {'city': 'Delhi'}, 'id': '00273974-d77a-4403-89eb-e1cc39f979dc', 'type': 'tool_call'}], usage_metadata={'input_tokens': 36, 'output_tokens': 7, 'total_tokens': 43, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='{\n  "city": "Delhi",\n  "weather": "haze",\n  "temperature": 291.2,\n  "unit": "kelvin"\n}', name='get_current_weather', id='b654cb45-17df-40f2-9470-76f7d8a6b171', tool_call_id='0027

In [8]:
await agent.ainvoke({"messages":[{"role":'user',"content":"Tell me more about city Jaiselmer?"}]})

{'messages': [HumanMessage(content='Tell me more about city Jaiselmer?', additional_kwargs={}, response_metadata={}, id='7aa27ddc-b519-4acb-bea6-a215e13830a9'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_wikipedia_summary', 'arguments': '{"query": "Jaiselmer"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--7b282d52-24c6-49a9-b2f8-ea35b6924459-0', tool_calls=[{'name': 'get_wikipedia_summary', 'args': {'query': 'Jaiselmer'}, 'id': '07a6fbf1-6055-4b2a-bcc2-3888d9a2fa32', 'type': 'tool_call'}], usage_metadata={'input_tokens': 38, 'output_tokens': 9, 'total_tokens': 47, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content="Jaisalmer (IPA: [d͡ʒɛːsəlmeːɾ] ), nicknamed The Golden city, is a city in the north-western Indian state of Rajasthan, located 575 kilometres (357 mi) west of the s

In [9]:
%%writefile mcpserver/mcpserver2.py

from mcp.server.fastmcp import FastMCP
import requests, json
import wikipedia

mcp = FastMCP("TredenceMCP")

@mcp.tool()
def get_current_weather(city:str)->dict:
    """ this funciton can be used to get current weather information"""
    api_key="6a8b0ac166a37e2b7a38e64416b3c3fe"
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}"
    response = requests.get(url)
    response = json.loads(response.content.decode())
    output = {"city":city,"weather":response['weather'][0]['description'],
              "temperature":response['main']['temp'], "unit":"kelvin"
              }
    return output

@mcp.tool()
def get_wikipedia_summary(query:str)->str:
    response = wikipedia.summary(query)
    return response


if __name__=="__main__":
    mcp.run(transport='stdio')


Writing mcpserver/mcpserver2.py


In [10]:
client = MultiServerMCPClient({"TredenceMCP":{"command":"python","args":["mcpserver/mcpserver2.py"],
                                              "transport":"stdio"}})


tools = await client.get_tools()
tools

[StructuredTool(name='get_current_weather', description=' this funciton can be used to get current weather information', args_schema={'properties': {'city': {'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_current_weatherArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7c4a22f28cc0>),
 StructuredTool(name='get_wikipedia_summary', args_schema={'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'get_wikipedia_summaryArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x7c4a22f28900>)]

In [11]:
agent = create_agent("google_genai:gemini-2.0-flash",tools)
await agent.ainvoke({"messages":[{"role":'user',"content":"what is the weather in Delhi?"}]})

{'messages': [HumanMessage(content='what is the weather in Delhi?', additional_kwargs={}, response_metadata={}, id='b46a3baa-3661-4788-901b-9700c2469f66'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_weather', 'arguments': '{"city": "Delhi"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--12135834-e63e-4fc7-a34c-9ff4d364021e-0', tool_calls=[{'name': 'get_current_weather', 'args': {'city': 'Delhi'}, 'id': '7d594495-55c3-4bee-9681-8612ff405062', 'type': 'tool_call'}], usage_metadata={'input_tokens': 36, 'output_tokens': 7, 'total_tokens': 43, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='{\n  "city": "Delhi",\n  "weather": "haze",\n  "temperature": 291.2,\n  "unit": "kelvin"\n}', name='get_current_weather', id='63573841-f811-4ac3-afd6-ea7e7bfdae56', tool_call_id='7d59